In [ ]:
# Configuration environnement Colab/Local
import sys, os, pathlib
import warnings
warnings.filterwarnings('ignore')

# Détection automatique
if 'google.colab' in str(get_ipython()):
    PROJECT_ROOT = pathlib.Path("/content/projet_ai")
    %cd /content/projet_ai
else:
    PROJECT_ROOT = pathlib.Path.cwd().parent if 'notebooks' in str(pathlib.Path.cwd()) else pathlib.Path.cwd()

# Ajouter src/ au path
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Créer dossiers de sortie
for folder in ["reports/figures", "reports/tables", "models"]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Fixer seeds
import numpy as np
np.random.seed(42)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Dossiers de sortie créés")

# Étape 2 : Entraînement des modèles

## APS Failure at Scania Trucks — Livraison 2

Trois modèles : Elastic Net (baseline), Random Forest + proximité, XGBoost + Optuna (A vs B).

**Métriques** : F1-Macro, AUPRC, MCC — jamais accuracy.

---


## 0. Configuration


In [ ]:
import os
import sys
import time
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# PROJECT_ROOT already defined
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.loader import load_aps_data, get_X_y, compute_total_cost, COST_FP, COST_FN
from src.utils.reproducibility import set_all_seeds, log_environment_info, setup_logging, get_cv_splitter
from src.evaluation.metrics import compute_all_metrics, print_metrics_report, find_optimal_threshold
from src.evaluation.plots import setup_plot_style, save_figure, COLORS

warnings.filterwarnings('ignore')
setup_logging(level=logging.INFO)
setup_plot_style()
%matplotlib inline

SEED = 42
set_all_seeds(SEED)
N_FOLDS = 5
cv = get_cv_splitter(n_splits=N_FOLDS, seed=SEED)

FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')
TABLES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'tables')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
for d in [FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
log_environment_info()
pd.show_versions()


## 1. Chargement des données


In [ ]:
train_df, test_df = load_aps_data(project_root=PROJECT_ROOT)
X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

n_pos = int(y_train.sum())
n_neg = int((y_train == 0).sum())
SCALE_POS_WEIGHT = n_neg / n_pos
SPW_CANDIDATES = [10, 30, int(round(SCALE_POS_WEIGHT)), 77]

print(f'Train {X_train.shape} | pos={n_pos} ({y_train.mean()*100:.2f}%)')
print(f'scale_pos_weight ≈ {SCALE_POS_WEIGHT:.1f}')


## 2. Utilitaires


In [ ]:
import joblib
from typing import Dict, List
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.metrics import make_scorer, matthews_corrcoef, average_precision_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.manifold import MDS
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

from src.preprocessing.pipeline import create_model_pipeline

scoring_cv = {
    'f1_macro': 'f1_macro',
    'auprc': make_scorer(average_precision_score, needs_proba=True),
    'mcc': make_scorer(matthews_corrcoef),
}
cv_results_all: List[Dict] = []
model_times: Dict[str, float] = {}


def compute_proximity_matrix(leaf_indices: np.ndarray) -> np.ndarray:
    n, n_trees = leaf_indices.shape
    prox = np.zeros((n, n), dtype=np.float64)
    for t in range(n_trees):
        leaves = leaf_indices[:, t]
        for leaf_id in np.unique(leaves):
            idx = np.where(leaves == leaf_id)[0]
            if len(idx) > 1:
                prox[np.ix_(idx, idx)] += 1.0
    prox /= n_trees
    np.fill_diagonal(prox, 1.0)
    return prox


def focal_loss_objective(gamma: float = 2.0, alpha: float = 0.25):
    def objective(y_pred, dtrain):
        y = dtrain.get_label()
        p = 1.0 / (1.0 + np.exp(-y_pred))
        p = np.clip(p, 1e-7, 1.0 - 1e-7)
        pt = np.where(y == 1, p, 1 - p)
        w = np.where(y == 1, alpha, 1 - alpha)
        mod = (1.0 - pt) ** gamma
        grad = w * mod * (p - y)
        hess = np.maximum(w * mod * p * (1.0 - p), 1e-6)
        return grad, hess
    return objective


def run_xgb_cv_mcc(X, y, params, use_focal: bool = False) -> float:
    mcc_scores = []
    for tr, val in cv.split(X, y):
        dtrain = xgb.DMatrix(X[tr], label=y[tr])
        dval = xgb.DMatrix(X[val], label=y[val])
        bp = {
            'max_depth': params['max_depth'],
            'eta': params['learning_rate'],
            'lambda': params['lambda'],
            'alpha': params['alpha'],
            'subsample': params['subsample'],
            'colsample_bytree': params['colsample_bytree'],
            'objective': 'binary:logistic',
            'seed': SEED,
            'verbosity': 0,
        }
        if not use_focal:
            bp['scale_pos_weight'] = params['scale_pos_weight']
        kw = {'num_boost_round': params.get('n_estimators', 300), 'evals': [(dval, 'val')], 'verbose_eval': False}
        if use_focal:
            bst = xgb.train(bp, dtrain, obj=focal_loss_objective(2.0, 0.25), **kw)
        else:
            bst = xgb.train(bp, dtrain, **kw)
        pred = (bst.predict(dval) >= 0.5).astype(int)
        mcc_scores.append(matthews_corrcoef(y[val], pred))
    return float(np.mean(mcc_scores))


def cv_summary(estimator, X, y, name: str) -> Dict:
    t0 = time.time()
    scores = cross_validate(estimator, X, y, cv=cv, scoring=scoring_cv, n_jobs=-1)
    elapsed = time.time() - t0
    model_times[name] = elapsed
    row = {
        'Modèle': name,
        'F1-Macro (CV)': scores['test_f1_macro'].mean(),
        'F1-Macro std': scores['test_f1_macro'].std(),
        'AUPRC (CV)': scores['test_auprc'].mean(),
        'AUPRC std': scores['test_auprc'].std(),
        'MCC (CV)': scores['test_mcc'].mean(),
        'MCC std': scores['test_mcc'].std(),
        'Temps (s)': elapsed,
    }
    cv_results_all.append(row)
    return row


## 3. Modèle 1 — Régression logistique Elastic Net

Pipeline : `SimpleImputer` → `StandardScaler` → `LogisticRegression(elasticnet, saga, class_weight='balanced')`


In [ ]:
print('='*60)
print('MODÈLE 1 : Elastic Net')
print('='*60)
t0 = time.time()

log_reg = LogisticRegression(
    penalty='elasticnet', solver='saga', class_weight='balanced',
    max_iter=5000, random_state=SEED,
)
pipe_lr = create_model_pipeline(log_reg, with_smote=False, scaling=True)

param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.0, 0.3, 0.5, 0.7, 1.0],
}

grid_lr = GridSearchCV(
    pipe_lr, param_grid, cv=cv, scoring=scoring_cv, refit='mcc', n_jobs=-1, verbose=1,
)
grid_lr.fit(X_train, y_train)
model_times['Elastic Net'] = time.time() - t0

best_lr = grid_lr.best_estimator_
print(f'Meilleurs params : {grid_lr.best_params_}')
print(f'CV MCC (refit) : {grid_lr.best_score_:.4f}')

cv_summary(best_lr, X_train, y_train, 'Elastic Net')
joblib.dump(best_lr, os.path.join(MODELS_DIR, 'logreg_elasticnet.pkl'))
joblib.dump(grid_lr, os.path.join(MODELS_DIR, 'logreg_gridsearch.pkl'))
print('Sauvegardé : models/logreg_elasticnet.pkl')


## 4. Modèle 2 — Random Forest + proximité + MDS

- RF entraîné sur **60k** | proximité/MDS sur **5k** (stratifié)


In [ ]:
print('='*60)
print('MODÈLE 2 : Random Forest')
print('='*60)
t0 = time.time()

rf_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_leaf=5,
        class_weight='balanced', random_state=SEED, n_jobs=-1,
    )),
])
rf_model.fit(X_train, y_train)
model_times['Random Forest'] = time.time() - t0

cv_summary(rf_model, X_train, y_train, 'Random Forest')
joblib.dump(rf_model, os.path.join(MODELS_DIR, 'random_forest.pkl'))
print('Sauvegardé : models/random_forest.pkl')


In [ ]:
PROXIMITY_N = 5000
rng = np.random.RandomState(SEED)
idx_pos = np.where(y_train.values == 1)[0]
idx_neg = np.where(y_train.values == 0)[0]
n_pos_s = min(len(idx_pos), max(1, int(PROXIMITY_N * y_train.mean())))
n_neg_s = PROXIMITY_N - n_pos_s
idx_sample = np.concatenate([
    rng.choice(idx_pos, n_pos_s, replace=False),
    rng.choice(idx_neg, n_neg_s, replace=False),
])
rng.shuffle(idx_sample)

X_sub = X_train.iloc[idx_sample]
y_sub = y_train.iloc[idx_sample]
X_sub_imp = rf_model.named_steps['imputer'].transform(X_sub)
rf_fitted = rf_model.named_steps['rf']

print(f'Proximité sur {len(idx_sample)} points (RF entraîné sur {len(X_train)})')
leaf_idx = rf_fitted.apply(X_sub_imp)
proximity = compute_proximity_matrix(leaf_idx)
dissim = 1.0 - proximity

mds = MDS(n_components=2, dissimilarity='precomputed', random_state=SEED, n_init=4, max_iter=300)
embedding = mds.fit_transform(dissim)

y_proba_sub = rf_fitted.predict_proba(X_sub_imp)[:, 1]
uncertainty = np.abs(y_proba_sub - 0.5)

iso = IsolationForest(contamination=0.05, random_state=SEED)
is_outlier = iso.fit_predict(embedding) == -1

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(embedding[~is_outlier, 0], embedding[~is_outlier, 1],
           c=COLORS['primary'], alpha=0.3, s=15, label='Inliers')
ax.scatter(embedding[is_outlier, 0], embedding[is_outlier, 1],
           c=COLORS['danger'], alpha=0.9, s=45, edgecolors='k', linewidths=0.4, label='Outliers')
ax.set_title('MDS 2D (1 - proximité RF) — outliers IsolationForest')
ax.legend()
plt.tight_layout()
save_figure(fig, 'rf_mds_proximity_outliers', save_dir=FIGURES_DIR)
plt.show()

outlier_df = X_sub.copy()
outlier_df['y_true'] = y_sub.values
outlier_df['proba'] = y_proba_sub
outlier_df['uncertainty'] = uncertainty
outlier_df['is_outlier'] = is_outlier

global_med = X_train.median()
out_med = outlier_df.loc[is_outlier, X_train.columns].median()
z_shift = ((out_med - global_med) / (X_train.std() + 1e-8)).abs().sort_values(ascending=False)
print('Top capteurs décalés (outliers):')
display(z_shift.head(10).to_frame('z_shift'))

print('''
**Analyse** : outliers = faible proximité RF + isolement MDS.
- Incertitude élevée → frontière floue.
- Capteurs atypiques → patterns rares / bruit.
- Erreurs concentrées → cas limites du classifieur.
''')


## 5. Modèle 3 — XGBoost + Optuna (50 essais × 2 stratégies)


In [ ]:
_imputer = SimpleImputer(strategy='median')
X_tr_imp = _imputer.fit_transform(X_train)
X_te_imp = _imputer.transform(X_test)
y_arr = y_train.values

N_TRIALS = 50
STORAGE = f"sqlite:///{os.path.join(MODELS_DIR, 'optuna_study.db')}"


def make_objective(use_focal: bool):
    def objective(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'lambda': trial.suggest_float('lambda', 1e-3, 10.0, log=True),
            'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        }
        if not use_focal:
            params['scale_pos_weight'] = trial.suggest_categorical('scale_pos_weight', SPW_CANDIDATES)
        return run_xgb_cv_mcc(X_tr_imp, y_arr, params, use_focal=use_focal)
    return objective

print('Optuna Stratégie A (scale_pos_weight)...')
t0 = time.time()
study_a = optuna.create_study(
    direction='maximize', sampler=TPESampler(seed=SEED),
    study_name='xgb_strategy_a', storage=STORAGE, load_if_exists=True,
)
study_a.optimize(make_objective(False), n_trials=N_TRIALS, show_progress_bar=True)
time_a = time.time() - t0

print('Optuna Stratégie B (Focal Loss γ=2, α=0.25)...')
t0 = time.time()
study_b = optuna.create_study(
    direction='maximize', sampler=TPESampler(seed=SEED),
    study_name='xgb_strategy_b', storage=STORAGE, load_if_exists=True,
)
study_b.optimize(make_objective(True), n_trials=N_TRIALS, show_progress_bar=True)
time_b = time.time() - t0

print(f'A best MCC CV={study_a.best_value:.4f} | B best MCC CV={study_b.best_value:.4f}')
use_focal = study_b.best_value > study_a.best_value
best_study = study_b if use_focal else study_a
best_params = best_study.best_params
strategy_label = 'B (Focal)' if use_focal else 'A (SPW)'
model_times['XGBoost'] = time_a + time_b


In [ ]:
# 📈 GRAPHIQUES DE CONVERGENCE OPTUNA
import optuna.visualization as vis

vis.plot_optimization_history(study_strategy_a).write_image(
    PROJECT_ROOT / "reports/figures/07_optuna_history.png", scale=2
)
vis.plot_param_importances(study_strategy_a).write_image(
    PROJECT_ROOT / "reports/figures/08_optuna_importance.png", scale=2
)
vis.plot_slice(study_strategy_a, params=['max_depth', 'learning_rate']).write_image(
    PROJECT_ROOT / "reports/figures/09_optuna_slice.png", scale=2
)
print("✅ Graphiques Optuna sauvegardés")

In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice

def save_optuna_fig(fig, fname):
    path = os.path.join(FIGURES_DIR, fname)
    try:
        fig.write_image(path)
        print(f'  PNG: {path}')
    except Exception as e:
        html_path = path.replace('.png', '.html')
        fig.write_html(html_path)
        print(f'  HTML (kaleido manquant): {html_path} — {e}')
    return fig

for study, tag, title in [
    (study_a, 'strategy_a', 'Stratégie A — scale_pos_weight'),
    (study_b, 'strategy_b', 'Stratégie B — Focal Loss'),
]:
    fh = plot_optimization_history(study)
    fh.update_layout(title=f'Historique Optuna — {title}')
    save_optuna_fig(fh, f'optuna_history_{tag}.png')
    save_optuna_fig(plot_param_importances(study), f'optuna_importance_{tag}.png')
    fs = plot_slice(study, params=['max_depth', 'learning_rate'])
    fs.update_layout(title=f'Slice — {title}')
    save_optuna_fig(fs, f'optuna_slice_{tag}.png')

plot_optimization_history(study_a).show()
plot_param_importances(study_a).show()
plot_slice(study_a, params=['max_depth', 'learning_rate']).show()


In [ ]:
# Entraînement final meilleure stratégie
params = best_params.copy()
n_rounds = params.pop('n_estimators', 300)
dtrain = xgb.DMatrix(X_tr_imp, label=y_arr)
bp = {
    'max_depth': params['max_depth'], 'eta': params['learning_rate'],
    'lambda': params['lambda'], 'alpha': params['alpha'],
    'subsample': params['subsample'], 'colsample_bytree': params['colsample_bytree'],
    'objective': 'binary:logistic', 'seed': SEED, 'verbosity': 0,
}
bp = base_params.copy()

if not use_focal:
    bp['scale_pos_weight'] = params.get('scale_pos_weight', 59)
    print(f"📊 Stratégie A: scale_pos_weight = {bp['scale_pos_weight']}")
else:
    bp['objective'] = custom_focal_loss_objective
    print(" Stratégie B: Focal Loss (gamma=2.0, alpha=0.25)")

# Entraînement TOUJOURS exécuté (hors if/else)
bst = xgb.train(
    bp, dtrain, num_boost_round=n_rounds,
    evals=[(dval, 'val')], early_stopping_rounds=20, verbose_eval=False
)
print(f"✅ Modèle entraîné | Best Score: {bst.best_score:.4f}")

xgb_pack = {
    'booster': bst, 'imputer': _imputer, 'params': best_params,
    'use_focal': use_focal, 'strategy': strategy_label,
}
joblib.dump(xgb_pack, os.path.join(MODELS_DIR, 'xgboost_best.pkl'))

xgb_cv_row = {
    'Modèle': f'XGBoost {strategy_label}',
    'F1-Macro (CV)': np.nan,
    'AUPRC (CV)': np.nan,
    'MCC (CV)': best_study.best_value,
    'MCC std': 0,
    'Temps (s)': model_times['XGBoost'],
}
cv_results_all.append(xgb_cv_row)
print(f'Sauvegardé xgboost_best.pkl — stratégie {strategy_label}')


## 6. Tableau comparatif et conclusion


In [ ]:
cv_df = pd.DataFrame(cv_results_all)
display(cv_df.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = cv_df.dropna(subset=['MCC (CV)'])
x = np.arange(len(plot_df))
w = 0.25
ax.bar(x - w, plot_df['F1-Macro (CV)'], w, label='F1-Macro')
ax.bar(x, plot_df['AUPRC (CV)'], w, label='AUPRC')
ax.bar(x + w, plot_df['MCC (CV)'], w, label='MCC')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['Modèle'], rotation=20, ha='right')
ax.set_ylim(0, 1.05)
ax.set_title('Comparaison CV des modèles')
ax.legend()
plt.tight_layout()
save_figure(fig, 'model_cv_comparison', save_dir=FIGURES_DIR)
plt.show()

cv_df.to_csv(os.path.join(TABLES_DIR, 'model_cv_comparison.csv'), index=False)
best = cv_df.loc[cv_df['MCC (CV)'].idxmax()]
print(f'\n🏆 Meilleur modèle (MCC CV) : {best["Modèle"]} — MCC={best["MCC (CV)"]:.4f}')
print('Prochaine étape : 03_evaluation_calibration.ipynb')
